In [1]:
import config
from pathlib import Path

from llama_index.core import SimpleDirectoryReader

In [2]:
pdf_files = list(config.DATA_DIR.glob("*.pdf"))

for i, pdf in enumerate(pdf_files):
    print(i, repr(pdf.name))

0 'Guideline for the pharmacological treatment of hypertension in adults.pdf'
1 'malaria_book.pdf'
2 'WHO_Hypertension_Guideline_2021.pdf'
3 '_OceanofPDF.com_Harrisons_principals_of_internal_medicine_-_Tinsley_R_harrison.pdf'


In [3]:
malaria_pdf = pdf_files[1]

print("Selected PDF:")
print(malaria_pdf.name)

Selected PDF:
malaria_book.pdf


In [4]:
documents = SimpleDirectoryReader(
    input_files=[str(malaria_pdf)]
).load_data()

print("Total documents:", len(documents))

Total documents: 317


In [5]:
for i in [0, 10, 50, 99]:
    print("=" * 80)
    print("Page:", i + 1)
    print(documents[i].text[:1000])

Page: 1
Third edition
FOR THE TREATMENT
OF MALARIA
GUIDELINES
Page: 11
Malaria case management, consisting of early diagnosis and prompt effective 
treatment, remains a vital component of malaria control and elimination strategies. 
This third edition of the WHO Guidelines for the treatment of malaria contains 
updated recommendations based on new evidence particularly related to dosing 
in children, and also includes recommendations on the use of drugs to prevent 
malaria in groups at high risk. 
Core principles
The following core principles were used by the Guidelines Development Group 
that drew up these Guidelines. 
1. Early diagnosis and prompt, effective treatment of malaria
Uncomplicated falciparum malaria can progress rapidly to severe forms of the 
disease, especially in people with no or low immunity, and severe falciparum 
malaria is almost always fatal without treatment. Therefore, programmes should 
ensure access to early diagnosis and prompt, effective treatment within 24

In [6]:
documents = documents[:100]

print("Documents used:", len(documents))

Documents used: 100


In [7]:
for i in [0, 10, 50, 99]:

    print("=" * 80)
    print(f"Page index: {i}")
    print(documents[i].text[:700])

Page index: 0
Third edition
FOR THE TREATMENT
OF MALARIA
GUIDELINES
Page index: 10
Malaria case management, consisting of early diagnosis and prompt effective 
treatment, remains a vital component of malaria control and elimination strategies. 
This third edition of the WHO Guidelines for the treatment of malaria contains 
updated recommendations based on new evidence particularly related to dosing 
in children, and also includes recommendations on the use of drugs to prevent 
malaria in groups at high risk. 
Core principles
The following core principles were used by the Guidelines Development Group 
that drew up these Guidelines. 
1. Early diagnosis and prompt, effective treatment of malaria
Uncomplicated falciparum malaria can progress rapidly to severe forms of the 
dis
Page index: 50
Treating uncomplicated P . falciparum malaria in special risk groups
First trimester of pregnancy
Treat pregnant women with uncomplicated P. falciparum malaria during the 
first trimester with 7 days o

In [8]:
from llama_index.core.node_parser import SentenceSplitter

CHUNK_SIZE = 500
CHUNK_OVERLAP = 50

splitter = SentenceSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)

nodes = splitter.get_nodes_from_documents(documents)

print("Chunk size:", CHUNK_SIZE)
print("Chunk overlap:", CHUNK_OVERLAP)
print("Number of chunks:", len(nodes))

Chunk size: 500
Chunk overlap: 50
Number of chunks: 162


In [9]:
print("Sample chunk:")
print(nodes[0].text[:1000])

print("\nMetadata:")
print(nodes[0].metadata)

Sample chunk:
Third edition
FOR THE TREATMENT
OF MALARIA
GUIDELINES

Metadata:
{'page_label': 'i', 'file_name': 'malaria_book.pdf', 'file_path': 'C:\\Users\\bodyn\\OneDrive\\Desktop\\LLM\\Day1\\Data\\malaria_book.pdf', 'file_type': 'application/pdf', 'file_size': 2540174, 'creation_date': '2026-08-17', 'last_modified_date': '2026-08-17'}


In [10]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

embed_model_1 = HuggingFaceEmbedding(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding Model 1 ready")

2026-08-17 21:17:42,864 - INFO - TensorFlow version 2.21.0 available.
2026-08-17 21:17:52,964 - WARNING - From c:\Users\bodyn\AppData\Local\Programs\Python\Python313\Lib\site-packages\tf_keras\src\losses.py:2976: The name tf.losses.sparse_softmax_cross_entropy is deprecated. Please use tf.compat.v1.losses.sparse_softmax_cross_entropy instead.

2026-08-17 21:17:53,687 - INFO - Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2


Embedding Model 1 ready


In [11]:
test_embedding = embed_model_1.get_text_embedding(
    "What is the treatment for malaria?"
)

print("Embedding dimension:", len(test_embedding))
print("First 10 values:", test_embedding[:10])

Embedding dimension: 384
First 10 values: [-0.026081407442688942, 0.09340314567089081, -0.01612800732254982, -0.06662168353796005, 0.004994289018213749, 0.000736494199372828, -0.006952063180506229, 0.1033332571387291, 0.004026738461107016, 0.05262570083141327]


In [12]:
from llama_index.core import VectorStoreIndex

index_1 = VectorStoreIndex(
    nodes,
    embed_model=embed_model_1
)

print("Vector index 1 built successfully!")

Vector index 1 built successfully!


In [13]:
retriever_1 = index_1.as_retriever(
    similarity_top_k=10
)

print("Retriever 1 ready!")

Retriever 1 ready!


In [14]:
questions = [
    "According to the document, what is the core principle regarding how antimalarial agents should be used to reduce the spread of drug resistance?",

    'What is the definition of "uncomplicated malaria" according to these guidelines?',

    "What is the recommended diagnostic approach for all cases of suspected malaria?",

    "What is the recommended duration of treatment for ACT (Artemisinin-based Combination Therapy) regimens?",

    "What is the recommended treatment for pregnant women with uncomplicated P. falciparum malaria during the first trimester?",

    "What is the difference between recrudescence, relapse, and re-infection in malaria?",

    "What is the recommended primaquine dose for reducing transmission of P. falciparum in low-transmission areas, and who should not receive it?",

    "According to the guidelines, what is the recommended treatment for infants weighing less than 5 kg with uncomplicated P. falciparum malaria?",

    "What are the five recommended ACTs for treating uncomplicated P. falciparum malaria, and what is the quality of evidence supporting this recommendation?",

    "What is the recommended treatment for severe malaria, and what is the revised dose recommendation for parenteral artesunate in young children?"
]

print("Number of questions:", len(questions))

Number of questions: 10


In [ ]:
evidence_phrases = [
    "To reduce the spread of drug resistance",

    "Uncomplicated malaria. Symptomatic malaria parasitaemia",

    "All cases of suspected malaria should have a parasitological test",

    "ACT regimens should provide 3 days",

    "Treat pregnant women with uncomplicated P. falciparum malaria during the first trimester",

    "Recrudescence. Recurrence of asexual parasitaemia",

    "0.25mg/kg bw primaquine",

    "Treat infants weighing < 5kg with uncomplicated P. falciparum malaria",

    "Treat children and adults with uncomplicated P. falciparum malaria",

    "Treat adults and children with severe malaria"
]

In [17]:
expected_nodes_1 = {}

for q_num, phrase in enumerate(evidence_phrases, 1):

    matches = []

    for node in nodes:

        if phrase.lower() in node.text.lower():
            matches.append(node)

    expected_nodes_1[q_num] = matches

    print("=" * 80)
    print(f"Question {q_num}")
    print(f"Evidence: {phrase}")
    print(f"Matching nodes: {len(matches)}")

    for node in matches:
        print(
            f"Node ID: {node.node_id} | "
            f"Page: {node.metadata.get('page_label')}"
        )

Question 1
Evidence: To reduce the spread of drug resistance
Matching nodes: 0
Question 2
Evidence: Uncomplicated malaria. Symptomatic malaria parasitaemia
Matching nodes: 1
Node ID: 353277e6-99bc-4bb5-85d4-d69e2ad12000 | Page: 5
Question 3
Evidence: All cases of suspected malaria should have a parasitological test
Matching nodes: 2
Node ID: 1edb8b22-e995-4f5e-8c5b-0c0693cd6648 | Page: 9
Node ID: e42906b1-b3b2-4105-96a3-87e4e013cdad | Page: 28
Question 4
Evidence: ACT regimens should provide 3 days
Matching nodes: 3
Node ID: 1edb8b22-e995-4f5e-8c5b-0c0693cd6648 | Page: 9
Node ID: 3e46030c-a75d-494a-9eae-665ed576f984 | Page: 32
Node ID: 1326cf85-45b2-4e6f-af73-4e57cd06f5e3 | Page: 34
Question 5
Evidence: Treat pregnant women with uncomplicated P. falciparum malaria during the first trimester
Matching nodes: 0
Question 6
Evidence: Recrudescence. Recurrence of asexual parasitaemia
Matching nodes: 1
Node ID: b1142d30-4291-4cd2-94fc-8a55330d3ef6 | Page: 4
Question 7
Evidence: 0.25mg/kg bw p

In [18]:
retrieval_results_1 = {}

for i, question in enumerate(questions, 1):

    results = retriever_1.retrieve(question)

    retrieval_results_1[i] = results

    print("=" * 100)
    print(f"Question {i}")
    print(question)
    print("-" * 100)

    for rank, result in enumerate(results, 1):

        print(
            f"[{rank}] "
            f"Score={result.score:.3f} | "
            f"Page={result.node.metadata.get('page_label')} | "
            f"Node={result.node.node_id}"
        )

        print(
            result.node.text[:300]
            .replace("\n", " ")
        )

        print()

Question 1
According to the document, what is the core principle regarding how antimalarial agents should be used to reduce the spread of drug resistance?
----------------------------------------------------------------------------------------------------
[1] Score=0.669 | Page=8 | Node=5a7301b4-bdd1-4a40-b4b7-6153c8b91d8a
Treatment should maximize  the likelihood of rapid clinical and parasitological cure and minimize transmission  from the treated infection. T o achieve this, dosage regimens should be based on  the patient’s weight and should provide effective concentrations of antimalarial  drugs for a sufficient t

[2] Score=0.663 | Page=16 | Node=a8088174-b75d-4c46-9b15-49e062b472eb
1.3 | SCOPE The Guidelines provide a framework for designing specific, detailed national  treatment protocols, taking into account local patterns of resistance to antimalarial  drugs and health service capacity.  The Guidelines provide evidence-based recommendations on:  •  the treatment of uncompli

[

In [19]:
def precision_at_k(results, relevant_ids, k):

    retrieved_ids = {
        result.node.node_id
        for result in results[:k]
    }

    relevant_retrieved = len(
        retrieved_ids.intersection(relevant_ids)
    )

    return relevant_retrieved / k

In [20]:
def hit_at_k(results, relevant_ids, k):

    retrieved_ids = {
        result.node.node_id
        for result in results[:k]
    }

    return int(
        bool(retrieved_ids.intersection(relevant_ids))
    )

In [21]:
precision_3 = []
precision_5 = []

hit_3 = []
hit_5 = []
hit_10 = []

for i in range(1, 11):

    results = retrieval_results_1[i]

    relevant_ids = {
        node.node_id
        for node in expected_nodes_1[i]
    }

    p3 = precision_at_k(
        results,
        relevant_ids,
        k=3
    )

    p5 = precision_at_k(
        results,
        relevant_ids,
        k=5
    )

    h3 = hit_at_k(
        results,
        relevant_ids,
        k=3
    )

    h5 = hit_at_k(
        results,
        relevant_ids,
        k=5
    )

    h10 = hit_at_k(
        results,
        relevant_ids,
        k=10
    )

    precision_3.append(p3)
    precision_5.append(p5)

    hit_3.append(h3)
    hit_5.append(h5)
    hit_10.append(h10)

    print(
        f"Question {i}: "
        f"Precision@3={p3:.2%} | "
        f"Precision@5={p5:.2%} | "
        f"Hit@3={h3} | "
        f"Hit@5={h5} | "
        f"Hit@10={h10}"
    )

Question 1: Precision@3=0.00% | Precision@5=0.00% | Hit@3=0 | Hit@5=0 | Hit@10=0
Question 2: Precision@3=0.00% | Precision@5=0.00% | Hit@3=0 | Hit@5=0 | Hit@10=0
Question 3: Precision@3=33.33% | Precision@5=20.00% | Hit@3=1 | Hit@5=1 | Hit@10=1
Question 4: Precision@3=33.33% | Precision@5=20.00% | Hit@3=1 | Hit@5=1 | Hit@10=1
Question 5: Precision@3=0.00% | Precision@5=0.00% | Hit@3=0 | Hit@5=0 | Hit@10=0
Question 6: Precision@3=33.33% | Precision@5=20.00% | Hit@3=1 | Hit@5=1 | Hit@10=1
Question 7: Precision@3=0.00% | Precision@5=0.00% | Hit@3=0 | Hit@5=0 | Hit@10=0
Question 8: Precision@3=0.00% | Precision@5=0.00% | Hit@3=0 | Hit@5=0 | Hit@10=0
Question 9: Precision@3=0.00% | Precision@5=0.00% | Hit@3=0 | Hit@5=0 | Hit@10=0
Question 10: Precision@3=66.67% | Precision@5=40.00% | Hit@3=1 | Hit@5=1 | Hit@10=1


In [22]:
print("=" * 70)

print(
    f"Average Precision@3 = "
    f"{sum(precision_3) / len(precision_3):.2%}"
)

print(
    f"Average Precision@5 = "
    f"{sum(precision_5) / len(precision_5):.2%}"
)

print(
    f"Hit@3 = "
    f"{sum(hit_3) / len(hit_3):.2%}"
)

print(
    f"Hit@5 = "
    f"{sum(hit_5) / len(hit_5):.2%}"
)

print(
    f"Hit@10 = "
    f"{sum(hit_10) / len(hit_10):.2%}"
)

Average Precision@3 = 16.67%
Average Precision@5 = 10.00%
Hit@3 = 40.00%
Hit@5 = 40.00%
Hit@10 = 40.00%


In [23]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

embed_model_2 = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)

print("Embedding Model 2 ready")

2026-08-17 21:29:17,225 - INFO - Load pretrained SentenceTransformer: BAAI/bge-small-en-v1.5
'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /BAAI/bge-small-en-v1.5/resolve/main/modules.json (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: 770e34f4-9533-484c-b0eb-da8283e93388)')' thrown while requesting HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/./modules.json
2026-08-17 21:29:28,347 - WARNING - '(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /BAAI/bge-small-en-v1.5/resolve/main/modules.json (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: 770e34f4-9533-484c-b0eb-da8283e93388)')' thrown while requesting HE

Embedding Model 2 ready


In [24]:
test_embedding_2 = embed_model_2.get_text_embedding(
    "What is the treatment for malaria?"
)

print("Embedding dimension:", len(test_embedding_2))

Embedding dimension: 384


In [25]:
from llama_index.core import VectorStoreIndex

index_2 = VectorStoreIndex(
    nodes,
    embed_model=embed_model_2
)

print("Vector index 2 built successfully!")

Vector index 2 built successfully!


In [26]:
retriever_2 = index_2.as_retriever(
    similarity_top_k=10
)

print("Retriever 2 ready!")

Retriever 2 ready!


In [27]:
retrieval_results_2 = {}

for i, question in enumerate(questions, 1):

    results = retriever_2.retrieve(question)

    retrieval_results_2[i] = results

    print("=" * 100)
    print(f"Question {i}")
    print(question)
    print("-" * 100)

    for rank, result in enumerate(results, 1):

        print(
            f"[{rank}] "
            f"Score={result.score:.3f} | "
            f"Page={result.node.metadata.get('page_label')} | "
            f"Node={result.node.node_id}"
        )

        print(
            result.node.text[:300]
            .replace("\n", " ")
        )

        print()

Question 1
According to the document, what is the core principle regarding how antimalarial agents should be used to reduce the spread of drug resistance?
----------------------------------------------------------------------------------------------------
[1] Score=0.804 | Page=96 | Node=01ff50c1-095d-455a-8eed-56adf5f3605a
Mass antimalarial drug administration has been used extensively in various forms  over the past 80 years. The objective is to provide therapeutic concentrations of  antimalarial drugs to as large a proportion of the population as possible in order  to cure any asymptomatic infections and also to pre

[2] Score=0.793 | Page=8 | Node=91a657a2-9200-4d11-a6fd-4e7edeb1fe31
Malaria case management, consisting of early diagnosis and prompt effective  treatment, remains a vital component of malaria control and elimination strategies.  This third edition of the WHO Guidelines for the treatment of malaria contains  updated recommendations based on new evidence particularly

[

In [28]:
precision_3_2 = []
precision_5_2 = []

hit_3_2 = []
hit_5_2 = []
hit_10_2 = []

for i in range(1, 11):

    results = retrieval_results_2[i]

    relevant_ids = {
        node.node_id
        for node in expected_nodes_1[i]
    }

    p3 = precision_at_k(
        results,
        relevant_ids,
        k=3
    )

    p5 = precision_at_k(
        results,
        relevant_ids,
        k=5
    )

    h3 = hit_at_k(
        results,
        relevant_ids,
        k=3
    )

    h5 = hit_at_k(
        results,
        relevant_ids,
        k=5
    )

    h10 = hit_at_k(
        results,
        relevant_ids,
        k=10
    )

    precision_3_2.append(p3)
    precision_5_2.append(p5)

    hit_3_2.append(h3)
    hit_5_2.append(h5)
    hit_10_2.append(h10)

print("=" * 70)

print(
    f"Model 2 Average Precision@3 = "
    f"{sum(precision_3_2) / len(precision_3_2):.2%}"
)

print(
    f"Model 2 Average Precision@5 = "
    f"{sum(precision_5_2) / len(precision_5_2):.2%}"
)

print(
    f"Model 2 Hit@3 = "
    f"{sum(hit_3_2) / len(hit_3_2):.2%}"
)

print(
    f"Model 2 Hit@5 = "
    f"{sum(hit_5_2) / len(hit_5_2):.2%}"
)

print(
    f"Model 2 Hit@10 = "
    f"{sum(hit_10_2) / len(hit_10_2):.2%}"
)

Model 2 Average Precision@3 = 20.00%
Model 2 Average Precision@5 = 12.00%
Model 2 Hit@3 = 40.00%
Model 2 Hit@5 = 40.00%
Model 2 Hit@10 = 40.00%
